# 06 - Advanced Logging Summary

## Purpose

Create a structured observability output for the Night Shift extension.

This notebook stores run evidence for the advanced delivery layer.

## Expected output

`vattenfall_dev.analytics.gold_pipeline_observability_summary`

## Main fields

- notebook name
- source table
- target table
- rows read
- rows written
- validation status
- execution timestamp
- comments

## Main idea

The pipeline should produce evidence that helps with review, debugging, and presentation.

In [0]:
import sys
from pyspark.sql import Row

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from utils.observability_utils import observability_record

In [0]:
catalog = "vattenfall_dev"
schema = "analytics"

target_table = f"{catalog}.{schema}.gold_pipeline_observability_summary"

night_shift_outputs = [
    {
        "notebook_name": "01_advanced_trust_audit",
        "source_table": "Silver and Gold validation inputs",
        "target_table": f"{catalog}.{schema}.gold_data_trust_audit",
    },
    {
        "notebook_name": "02_asset_incident_intelligence",
        "source_table": f"{catalog}.refined.silver_grid_events, {catalog}.refined.silver_asset_reference",
        "target_table": f"{catalog}.{schema}.gold_asset_incident_intelligence",
    },
    {
        "notebook_name": "03_weather_grid_risk_correlation",
        "source_table": f"{catalog}.{schema}.gold_weather_impact_summary, {catalog}.{schema}.gold_grid_incident_summary",
        "target_table": f"{catalog}.{schema}.gold_weather_grid_risk_correlation",
    },
    {
        "notebook_name": "04_market_operations_stress",
        "source_table": f"{catalog}.{schema}.gold_daily_market_summary, {catalog}.{schema}.gold_grid_incident_summary",
        "target_table": f"{catalog}.{schema}.gold_market_operations_stress",
    },
    {
        "notebook_name": "05_executive_risk_dashboard_view",
        "source_table": "Night Shift risk and stress outputs",
        "target_table": f"{catalog}.{schema}.vw_executive_energy_risk_dashboard",
    },
]

print("Target observability table:", target_table)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

observability_rows = []

for output in night_shift_outputs:
    target = output["target_table"]

    try:
        rows_written = spark.table(target).count()
        validation_status = "PASS" if rows_written > 0 else "FAIL"
        comments = "Output exists and returns rows." if rows_written > 0 else "Output exists but is empty."
    except Exception as error:
        rows_written = 0
        validation_status = "FAIL"
        comments = f"Output could not be read: {str(error)}"

    record = observability_record(
        notebook_name=output["notebook_name"],
        source_table=output["source_table"],
        target_table=target,
        rows_read=0,
        rows_written=rows_written,
        validation_status=validation_status,
        comments=comments,
    )

    observability_rows.append(record)

observability_schema = StructType([
    StructField("notebook_name", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("validation_status", StringType(), True),
    StructField("execution_timestamp", StringType(), True),
    StructField("comments", StringType(), True),
])

observability_df = spark.createDataFrame(observability_rows, schema=observability_schema)

display(observability_df)

In [0]:
(
    observability_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Written Night Shift observability summary:", target_table)
print("Rows written:", observability_df.count())

In [0]:
result_df = spark.table(target_table)

print("Rows in observability summary:", result_df.count())
print("Columns:", result_df.columns)

display(result_df)

print("Validation status distribution:")
result_df.groupBy("validation_status").count().show()